<div style="color: green; font-weight: bold">Part 1 Comment</div>
My solution is essentially equal for the first logical OR and the masked logical OR. My answer for the perfect match differs as it uses a different activation function for the way I used dot products but is effectively the same answer with a sign flip, and results in the correct answer the question is asking for.

The drawing of decision boundaries I did incorrectly but gets at the core idea of splitting up the decision boundaries by the classifiers and assigning them to the corners of the hypercube. (I thought I could start and stop the lines from the other boundaries, this is incorrect as per the example).

I did not include the one-hot equations as I misunderstood that the equations used for the one-hot were merely the prior perfect match and masked OR equations made earlier in the question. I needed to show the hypercube corners mapped to the one-hot equations for the neurons.

For the construction of the neural network I used the perfect match for layer 2, which is identical to the example solution. I however got the final layer wrong as I used perfect match again while the sample uses a masked OR for the final layer. Masked OR is the right answer here because it is running an OR on whether the classifier is in one of two regions that the classifier is contained in. There are 6 regions with 2 for each classifier so it is getting the exact region or ID in the sample. My exact match does not work in this instance, it would only work if I had one decision region for each classifier which I do not (I only have 1 for the circles, it would only work in that instance).

My drawing of the network is incorrect as my boundary drawing was incorrect, but the general idea of layers connecting to nodes and the equations used is correct. My extension of how to draw more and extend the network is essentially the explanation given in the sample. I also correctly identify the issue of overfitting if this classifier is extended for larger training sets, and that it will fail at a larger scale.


# Exercise 2

## Part 3 — commented feedback version

In [1]:
import numpy as np
from sklearn import datasets


In [2]:
class ReLULayer(object):
    def forward(self, input):
        # Remember the input for later backpropagation.
        # ReLU is applied element-wise: max(0, x).
        self.input = input
        relu = np.maximum(0.0, input)
        return relu

    def backward(self, upstream_gradient):
        # Chain rule:
        # d ReLU(x) / dx is 1 for x > 0 and 0 for x <= 0.
        downstream_gradient = upstream_gradient * (self.input > 0)
        return downstream_gradient

    def update(self, learning_rate):
        pass  # ReLU is parameter-free


<div style="color: green; font-weight: bold">Comment</div>

**ReLULayer:**
The ReLULayer implementation is correct and compact — the forward pass uses `np.maximum(0.0, input)`, and the backward pass is fully vectorized: the gradient is set to 0 for `input <= 0` (including `input == 0`, as in the sample solution). Functionally, this is equivalent to the sample solution.


In [3]:
class OutputLayer(object):
    def __init__(self, n_classes):
        self.n_classes = n_classes

    def forward(self, input):
        # Remember the input for later backpropagation.
        self.input = input

        # softmax:
        exp_values = np.exp(input)
        softmax = exp_values / np.sum(exp_values, axis=1, keepdims=True)
        return softmax

    def backward(self, predicted_posteriors, true_labels):
        # For softmax + cross-entropy, the derivative w.r.t. the logits is:
        # predicted_posteriors - one_hot(true_labels).
        #
        # We divide by batch_size because the mini-batch loss is interpreted
        # as the average loss over the mini-batch.
        batch_size = predicted_posteriors.shape[0]

        true_labels = np.asarray(true_labels, dtype=int)
        one_hot = np.zeros_like(predicted_posteriors)
        one_hot[np.arange(batch_size), true_labels] = 1.0

        downstream_gradient = (predicted_posteriors - one_hot) / batch_size
        return downstream_gradient

    def update(self, learning_rate):
        pass  # softmax is parameter-free


<div style="color: green; font-weight: bold">Comment</div>

**OutputLayer:**
The backward pass is correct (`predicted_posteriors - one_hot(true_labels)`, divided by the batch size). The forward pass is missing numerical stabilization — before applying `np.exp`, one should subtract, for example, `input - np.max(input, axis=1, keepdims=True)`. Otherwise, large logits may cause overflow.


In [4]:
class LinearLayer(object):
    def __init__(self, n_inputs, n_outputs):
        self.n_inputs  = n_inputs
        self.n_outputs = n_outputs

        # He initialization, suitable for ReLU networks.
        # The lecture used D_{l-1}+1 when the bias was absorbed into the
        # weight matrix by adding a constant input 1. Here the bias is separate,
        # but we use the same scale for consistency with that lecture notation.
        std = np.sqrt(2.0 / (self.n_inputs + 1))

        self.B = np.random.normal(0.0, std, size=(self.n_inputs, self.n_outputs))
        self.b = np.random.normal(0.0, std, size=(1, self.n_outputs))

        # Placeholders for gradients, filled during backward().
        self.grad_B = np.zeros_like(self.B)
        self.grad_b = np.zeros_like(self.b)

    def forward(self, input):
        # Remember the input for later backpropagation.
        self.input = input

        # Linear preactivation:
        # Z_tilde = Z_previous @ B + b
        preactivations = np.dot(input, self.B) + self.b
        return preactivations

    def backward(self, upstream_gradient):
        # upstream_gradient is dLoss / dZ_tilde for this layer.

        # Bias gradient:
        # Since b is added to every instance, we sum over the batch axis.
        self.grad_b = np.sum(upstream_gradient, axis=0, keepdims=True)

        # Weight gradient:
        # For a batch: dLoss/dB = input.T @ upstream_gradient
        self.grad_B = np.dot(self.input.T, upstream_gradient)

        # Downstream gradient for the preceding layer:
        # dLoss/dInput = upstream_gradient @ B.T
        downstream_gradient = np.dot(upstream_gradient, self.B.T)
        return downstream_gradient

    def update(self, learning_rate):
        # Gradient descent update for trainable parameters.
        self.B = self.B - learning_rate * self.grad_B
        self.b = self.b - learning_rate * self.grad_b


<div style="color: green; font-weight: bold">Comment</div>

**LinearLayer:**
The LinearLayer implementation is correct — the forward pass, the gradients (`input.T @ upstream_gradient`, sum for the bias), and the downstream propagation (`upstream_gradient @ B.T`) are all consistent, and the bias shape `(1, n_outputs)` works through broadcasting. The initialization uses a He-like standard deviation `sqrt(2/(n_inputs+1))` instead of the sample solution's scale `2.0/n_inputs`; both variants are runnable, but the sample solution is closer to the reference implementation.


In [5]:
class MLP(object):
    def __init__(self, n_features, layer_sizes):
        # Construct a multi-layer perceptron with ReLU activations in the
        # hidden layers and softmax output.
        #
        # n_features: number of input features
        # len(layer_sizes): number of linear layers
        # layer_sizes[k]: number of neurons in linear layer k
        # layer_sizes[-1]: number of output classes
        self.n_layers = len(layer_sizes)
        self.layers   = []

        # Create interior layers: Linear + ReLU.
        n_in = n_features
        for n_out in layer_sizes[:-1]:
            self.layers.append(LinearLayer(n_in, n_out))
            self.layers.append(ReLULayer())
            n_in = n_out

        # Create last linear layer + output softmax.
        n_out = layer_sizes[-1]
        self.layers.append(LinearLayer(n_in, n_out))
        self.layers.append(OutputLayer(n_out))

    def forward(self, X):
        # X is a mini-batch of instances.
        batch_size = X.shape[0]

        # Flatten the other dimensions of X in case instances are images.
        X = X.reshape(batch_size, -1)

        # Compute the forward pass. Each layer stores what it needs for
        # subsequent backpropagation.
        result = X
        for layer in self.layers:
            result = layer.forward(result)
        return result

    def backward(self, predicted_posteriors, true_classes):
        # Start backpropagation at the output layer.
        upstream_gradient = self.layers[-1].backward(predicted_posteriors, true_classes)

        # Propagate the gradient through the remaining layers in reverse order.
        for layer in reversed(self.layers[:-1]):
            upstream_gradient = layer.backward(upstream_gradient)

    def update(self, X, Y, learning_rate):
        posteriors = self.forward(X)
        self.backward(posteriors, Y)

        for layer in self.layers:
            layer.update(learning_rate)

    def train(self, x, y, n_epochs, batch_size, learning_rate):
        N = len(x)
        n_batches = N // batch_size

        for i in range(n_epochs):
            # Reorder data for every epoch, i.e. sample mini-batches
            # without replacement within one epoch.
            permutation = np.random.permutation(N)

            for batch in range(n_batches):
                # Create mini-batch.
                start = batch * batch_size
                x_batch = x[permutation[start:start + batch_size]]
                y_batch = y[permutation[start:start + batch_size]]

                # Perform one forward pass, one backward pass,
                # and one parameter update.
                self.update(x_batch, y_batch, learning_rate)


<div style="color: green; font-weight: bold">Comment</div>

**MLP class:**
The backpropagation corresponds to the sample solution.
Readability: `reversed(self.layers[:-1])` is clear and correct.
Efficiency: The implementation is fully vectorized and does not use per-sample Python loops.


In [6]:
if __name__ == "__main__":

    #np.random.seed(0)  # For reproducibility.

    # Set training/test set size.
    N = 2000

    # Create training and test data.
    X_train, Y_train = datasets.make_moons(N, noise=0.05, random_state=0)
    X_test,  Y_test  = datasets.make_moons(N, noise=0.05, random_state=1)
    n_features = 2
    n_classes  = 2

    # Standardize features to be in [-1, 1], using training-set statistics.
    offset  = X_train.min(axis=0)
    scaling = X_train.max(axis=0) - offset
    X_train = ((X_train - offset) / scaling - 0.5) * 2.0
    X_test  = ((X_test  - offset) / scaling - 0.5) * 2.0

    # Set hyperparameters.
    n_epochs = 5
    batch_size = 200
    learning_rate = 0.05

    # Compare the four networks requested in the exercise sheet.
    architectures = [
        [2, 2, n_classes],
        [3, 3, n_classes],
        [5, 5, n_classes],
        [30, 30, n_classes],
    ]

    for layer_sizes in architectures:
        network = MLP(n_features, layer_sizes)

        # Train.
        network.train(X_train, Y_train, n_epochs, batch_size, learning_rate)

        # Test.
        predicted_posteriors = network.forward(X_test)

        # Winner-takes-all rule.
        predicted_classes = np.argmax(predicted_posteriors, axis=1)

        # Error rate.
        error_rate = np.mean(predicted_classes != Y_test)

        print("architecture:", layer_sizes, "error rate:", error_rate)


architecture: [2, 2, 2] error rate: 0.18
architecture: [3, 3, 2] error rate: 0.5
architecture: [5, 5, 2] error rate: 0.187
architecture: [30, 30, 2] error rate: 0.1085


<div style="color: green; font-weight: bold">Comment</div>

**Training and testing:** The loop over several architectures is a useful extension compared to the sample solution, because it makes the influence of the network size directly comparable. The main weakness is `n_epochs = 5`: the sample solution trains for much longer, and 5 epochs may lead to undertraining. For more meaningful feedback on the architectures, the number of epochs should be increased significantly, for example toward the value used in the sample solution. The fixed `random_state` values are a positive choice because they make the results more reproducible.


Different architectures produce different error rates. In general, increasing the number of neurons improves the error rate.
